# Chapter 1: Correlation, Association & the Yule–Simpson Paradox

## Reusable Analysis Template

This notebook implements:
1. Pearson correlation & linear/multiple regression
2. 2×2 contingency table analysis (RD, RR, OR)
3. Fisher's exact test & Chi-square test
4. Simpson's Paradox detection
5. Stratified subgroup analysis
6. Specification search over covariate subsets

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SECTION 1: Pearson Correlation & Linear Regression
# ============================================================

def pearson_correlation(z, y):
    """Compute Pearson correlation between two arrays."""
    rho = np.corrcoef(z, y)[0, 1]
    print(f'Pearson correlation ρ_ZY = {rho:.4f}')
    return rho

def simple_vs_multiple_regression(df, treatment, outcome, covariates):
    """Compare simple regression vs. multiple regression coefficients.
    Demonstrates sign-flip phenomenon (LaLonde-style).
    """
    import statsmodels.api as sm

    # Simple regression (no controls)
    X_simple = sm.add_constant(df[treatment])
    model_simple = sm.OLS(df[outcome], X_simple).fit()
    beta_simple = model_simple.params[treatment]

    # Multiple regression (all controls)
    X_multi = sm.add_constant(df[[treatment] + covariates])
    model_multi = sm.OLS(df[outcome], X_multi).fit()
    beta_multi = model_multi.params[treatment]

    print(f'Simple regression β ({treatment}):  {beta_simple:.4f} (p={model_simple.pvalues[treatment]:.4f})')
    print(f'Multiple regression β ({treatment}): {beta_multi:.4f} (p={model_multi.pvalues[treatment]:.4f})')

    if np.sign(beta_simple) != np.sign(beta_multi):
        print('⚠️  SIGN FLIP detected! Covariates reversed the direction of association.')
    else:
        print('No sign flip detected.')

    return {'simple_beta': beta_simple, 'multi_beta': beta_multi}

In [ ]:
# ============================================================
# SECTION 2: 2×2 Contingency Table Analysis
# ============================================================

def contingency_table_analysis(table, labels=('Z=1', 'Z=0')):
    """
    Given a 2×2 table as [[n11, n10], [n01, n00]],
    compute RD, RR, OR and run Fisher's exact & Chi-square tests.
    """
    table = np.array(table, dtype=float)
    n11, n10 = table[0]
    n01, n00 = table[1]

    # Risk (probability of Y=1) under each treatment
    p1 = n11 / (n11 + n10)  # P(Y=1 | Z=1)
    p0 = n01 / (n01 + n00)  # P(Y=1 | Z=0)

    rd = p1 - p0
    rr = p1 / p0 if p0 > 0 else float('inf')
    or_val = (n11 * n00) / (n10 * n01) if (n10 * n01) > 0 else float('inf')

    # Fisher's exact test
    fisher = stats.fisher_exact(table.astype(int))
    # Chi-square test
    chi2, chi2_p, _, _ = stats.chi2_contingency(table.astype(int), correction=False)

    print(f'Table: {labels[0]} vs {labels[1]}')
    print(f'  P(Y=1|Z=1) = {p1:.4f}  |  P(Y=1|Z=0) = {p0:.4f}')
    print(f'  Risk Difference (RD) = {rd:+.4f}')
    print(f'  Risk Ratio (RR)       = {rr:.4f}')
    print(f'  Odds Ratio (OR)       = {or_val:.4f}')
    print(f'  Fisher exact p-value  = {fisher[1]:.6f}')
    print(f'  Chi-square p-value    = {chi2_p:.6f}')
    print()

    return {'rd': rd, 'rr': rr, 'or': or_val,
            'fisher_p': fisher[1], 'chi2_p': chi2_p}

In [ ]:
# ============================================================
# SECTION 3: Simpson's Paradox Detector
# ============================================================

def detect_simpsons_paradox(marginal_table, subgroup_tables):
    """
    Detects Simpson's Paradox by comparing the sign of the marginal RD
    to the signs of all subgroup RDs.

    Parameters:
    - marginal_table: 2×2 array for the whole population
    - subgroup_tables: list of 2×2 arrays, one per subgroup
    """
    marginal_rd = contingency_table_analysis(marginal_table)['rd']
    marginal_sign = np.sign(marginal_rd)

    subgroup_signs = []
    for i, sg_table in enumerate(subgroup_tables):
        sg_result = contingency_table_analysis(sg_table)
        subgroup_signs.append(np.sign(sg_result['rd']))

    all_subgroup_same = all(s == subgroup_signs[0] for s in subgroup_signs)
    all_opposite_to_marginal = all(s != marginal_sign for s in subgroup_signs)

    if all_subgroup_same and all_opposite_to_marginal:
        print('🚨 SIMPSON\'S PARADOX DETECTED!')
        print(f'  Marginal RD sign: {marginal_sign:+.0f}')
        print(f'  All subgroup RD signs: {[f\"{s:+.0f}\" for s in subgroup_signs]}')
        print(f'  The marginal association reverses ALL conditional associations.')
    elif marginal_sign != subgroup_signs[0]:
        print('⚠️  Partial sign reversal detected (not full Simpson\'s Paradox).')
    else:
        print('✅ No Simpson\'s Paradox detected.')

    return {'marginal_rd': marginal_rd,
            'subgroup_rds': [contingency_table_analysis(t)['rd'] for t in subgroup_tables]}

In [ ]:
# ============================================================
# SECTION 4: Kidney Stone Example (from the chapter)
# ============================================================

print('='*60)
print('KIDNEY STONE EXAMPLE')
print('='*60)

# Marginal (whole population)
kidney_marginal = [[273, 77], [289, 61]]

# Subgroup: smaller stones (X=1)
kidney_small = [[81, 6], [234, 36]]

# Subgroup: larger stones (X=0)
kidney_large = [[192, 71], [55, 25]]

detect_simpsons_paradox(kidney_marginal, [kidney_small, kidney_large])

In [ ]:
# ============================================================
# SECTION 5: Berkeley Admissions Example
# ============================================================

print('='*60)
print('BERKELEY ADMISSIONS EXAMPLE')
print('='*60)

# Department-wise data: [[Male Admitted, Male Rejected], [Female Admitted, Female Rejected]]
berkeley_depts = {
    'A': [[512, 313], [89, 19]],
    'B': [[353, 207], [17, 8]],
    'C': [[120, 205], [202, 391]],
    'D': [[138, 279], [131, 244]],
    'E': [[53, 138], [94, 299]],
    'F': [[22, 351], [24, 317]],
}

# Aggregate (marginal) table
male_total_admit = sum(d[0][0] for d in berkeley_depts.values())
male_total_reject = sum(d[0][1] for d in berkeley_depts.values())
female_total_admit = sum(d[1][0] for d in berkeley_depts.values())
female_total_reject = sum(d[1][1] for d in berkeley_depts.values())
berkeley_marginal = [[male_total_admit, male_total_reject],
                     [female_total_admit, female_total_reject]]

print('\n--- MARGINAL (aggregated) ---')
contingency_table_analysis(berkeley_marginal)

print('--- BY DEPARTMENT ---')
dept_results = []
for dept, tbl in berkeley_depts.items():
    print(f'Department {dept}:')
    r = contingency_table_analysis(tbl)
    dept_results.append((dept, r))

In [ ]:
# ============================================================
# SECTION 6: Specification Search (Problem 1.4)
# ============================================================

def specification_search(df, treatment, outcome, covariates):
    """
    Run all 2^p subsets of covariates in linear regression,
    classify treatment coefficient as positive sig, negative sig, or not sig.
    """
    import statsmodels.api as sm

    n_subsets = 2 ** len(covariates)
    results = []

    for k in range(len(covariates) + 1):
        for combo in combinations(covariates, k):
            cols = [treatment] + list(combo)
            X = sm.add_constant(df[cols])
            model = sm.OLS(df[outcome], X).fit()
            beta = model.params[treatment]
            pval = model.pvalues[treatment]

            if pval < 0.05 and beta > 0:
                category = 'positive_sig'
            elif pval < 0.05 and beta < 0:
                category = 'negative_sig'
            else:
                category = 'not_sig'

            results.append({
                'covariates': combo,
                'beta': beta,
                'p_value': pval,
                'category': category
            })

    df_results = pd.DataFrame(results)
    counts = df_results['category'].value_counts()

    print(f'Total regressions run: {len(results)}')
    print(f'  Positive & significant: {counts.get("positive_sig", 0)}')
    print(f'  Negative & significant: {counts.get("negative_sig", 0)}')
    print(f'  Not significant:         {counts.get("not_sig", 0)}')

    return df_results

## How to Use This Template

1. **Load your data** into a pandas DataFrame
2. Call `simple_vs_multiple_regression()` to check for sign flips
3. Build 2×2 tables and call `contingency_table_analysis()` for RD, RR, OR, and hypothesis tests
4. Use `detect_simpsons_paradox()` to compare marginal vs. subgroup associations
5. Use `specification_search()` to exhaustively check all covariate subsets